
# Notebook 07 — Genuine External Multi-Sensor Gating Benchmark

**Purpose.** This notebook provides a second, genuinely independent external benchmark for the paper
*Gated Sequential Learning: Certification Ladders for Binary Identification under Endogenous Experiment Availability*.

It uses the **UCI Daily and Sports Activities** dataset (Barshan & Altun, UCI ML Repository, DOI: `10.24432/C5C59F`):
- 19 activities
- 8 subjects
- 5 body-worn Xsens MTx units
- 9 axes per unit (3 accelerometer, 3 gyroscope, 3 magnetometer)
- 25 Hz sampling
- 5-second segments (125 time points each)

### Evidence-gated sensing ladder
The external replay treats access to additional body sensors as a resource-gated observation ladder:

1. **L1**: torso only  
2. **L2**: torso + right arm  
3. **L3**: torso + both arms  
4. **L4**: torso + both arms + right leg  
5. **L5**: all five sensor units

The proposed policy unlocks richer sensing irreversibly when accumulated evidence crosses stage thresholds.

### Experiments
1. Binary theorem-aligned slice: **ascending stairs vs descending stairs**
2. Six-class external benchmark
3. Proposed gated vs weak-only vs full-sensor oracle vs fixed schedule vs reversible posterior gate
4. Error/sample/sensor-cost trade-off
5. Per-class diagnostics and confusion matrix
6. Delta sensitivity
7. Final automatic ZIP of every result and figure

> **Scope note.** This is an offline resource-gated sensor-acquisition replay on real data. It demonstrates external instantiation and generalization of the framework; it is not a physical safety certification experiment.


In [ ]:

# Configuration
import os, sys, math, json, time, shutil, zipfile, warnings, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import logsumexp
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix

warnings.filterwarnings("ignore")

SMOKE_TEST = os.getenv("SMOKE_TEST", "0") == "1"
FULL_RUN = os.getenv("FULL_RUN", "0") == "1"

ROOT = Path.cwd()
DATA_DIR = ROOT / "notebook07_data"
RESULTS = ROOT / "notebook07_results"
FIGURES = ROOT / "notebook07_figures"
for p in (DATA_DIR, RESULTS, FIGURES):
    p.mkdir(parents=True, exist_ok=True)

UCI_URL = "https://archive.ics.uci.edu/static/public/256/daily%2Band%2Bsports%2Bactivities.zip"
UCI_DOI = "10.24432/C5C59F"

ACTIVITY_NAMES = {
    1:"sitting", 2:"standing", 3:"lying_back", 4:"lying_right",
    5:"ascending_stairs", 6:"descending_stairs", 7:"elevator_still",
    8:"elevator_moving", 9:"walking_parking", 10:"treadmill_walk_flat",
    11:"treadmill_walk_incline", 12:"treadmill_run", 13:"stepper",
    14:"cross_trainer", 15:"cycling_horizontal", 16:"cycling_vertical",
    17:"rowing", 18:"jumping", 19:"basketball"
}

# Six-class subset selected to mix static, locomotion, stair, and running behavior.
MULTI_ACTIVITIES = [1, 2, 5, 6, 9, 12]
BINARY_ACTIVITIES = [5, 6]

TRAIN_SUBJECTS = [1,2,3,4,5,6]
CAL_SUBJECTS = [7]
TEST_SUBJECTS = [8]

# Sensor blocks: 9 axes each, exactly matching the UCI documentation.
LEVEL_COLS = {
    1: list(range(0,9)),      # torso
    2: list(range(0,18)),     # + right arm
    3: list(range(0,27)),     # + left arm
    4: list(range(0,36)),     # + right leg
    5: list(range(0,45)),     # + left leg
}
LEVEL_COST = {1:1.0, 2:2.0, 3:3.0, 4:4.0, 5:5.0}
RHO = [0.20, 0.40, 0.60, 0.80]

if SMOKE_TEST:
    MAX_SEGMENTS_PER_SUBJECT = 8
    N_BINARY_TRIALS = 30
    N_MULTI_TRIALS = 15
    DELTAS = [0.05]
    MAX_STEPS = 40
elif FULL_RUN:
    MAX_SEGMENTS_PER_SUBJECT = 60
    N_BINARY_TRIALS = 2000
    N_MULTI_TRIALS = 500
    DELTAS = [0.10, 0.05, 0.02, 0.01]
    MAX_STEPS = 150
else:
    MAX_SEGMENTS_PER_SUBJECT = 30
    N_BINARY_TRIALS = 500
    N_MULTI_TRIALS = 150
    DELTAS = [0.05, 0.01]
    MAX_STEPS = 100

SEED = 20260914
rng = np.random.default_rng(SEED)

print({
    "SMOKE_TEST": SMOKE_TEST,
    "FULL_RUN": FULL_RUN,
    "segments_per_subject_activity": MAX_SEGMENTS_PER_SUBJECT,
    "binary_trials_per_class": N_BINARY_TRIALS,
    "multi_trials_per_class": N_MULTI_TRIALS,
    "deltas": DELTAS,
})


In [ ]:

# Dataset acquisition and segment indexing
import urllib.request

def ensure_real_dataset():
    """Download/extract the official UCI archive. Internet is required only for the first run."""
    zip_path = DATA_DIR / "daily_sports_activities.zip"
    extract_dir = DATA_DIR / "daily_sports_activities"
    marker = extract_dir / "data"

    if marker.exists():
        return extract_dir

    if not zip_path.exists():
        print(f"Downloading official UCI dataset ({UCI_DOI})...")
        urllib.request.urlretrieve(UCI_URL, zip_path)

    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

    # UCI archive structures occasionally add a wrapper folder; locate data/.
    candidates = list(extract_dir.rglob("data"))
    candidates = [p for p in candidates if p.is_dir() and any(p.glob("a*"))]
    if not candidates:
        raise FileNotFoundError("Could not locate UCI data/ directory after extraction.")
    if candidates[0] != marker:
        # Keep the real location; no need to move the archive.
        return candidates[0].parent
    return extract_dir

def parse_real_index(base_dir, activities):
    data_root_candidates = list(Path(base_dir).rglob("data"))
    data_root_candidates = [p for p in data_root_candidates if p.is_dir() and any(p.glob("a*"))]
    if not data_root_candidates:
        raise FileNotFoundError("No data/aXX/pY/sZZ.txt hierarchy found.")
    data_root = data_root_candidates[0]

    rows = []
    for aid in activities:
        for sid in range(1,9):
            files = sorted((data_root / f"a{aid:02d}" / f"p{sid}").glob("s*.txt"))
            files = files[:MAX_SEGMENTS_PER_SUBJECT]
            for fp in files:
                seg = int(fp.stem[1:])
                rows.append({"activity": aid, "subject": sid, "segment": seg, "path": str(fp)})
    return pd.DataFrame(rows)

def make_smoke_index(activities):
    rows = []
    for aid in activities:
        for sid in range(1,9):
            for seg in range(1, MAX_SEGMENTS_PER_SUBJECT+1):
                rows.append({"activity": aid, "subject": sid, "segment": seg, "path": ""})
    return pd.DataFrame(rows)

if SMOKE_TEST:
    INDEX = make_smoke_index(MULTI_ACTIVITIES)
    REAL_DATA = False
    print("SMOKE_TEST: using deterministic synthetic sensor segments with the real dataset schema.")
else:
    BASE = ensure_real_dataset()
    INDEX = parse_real_index(BASE, MULTI_ACTIVITIES)
    REAL_DATA = True
    print("Loaded official UCI Daily and Sports Activities archive.")

print(INDEX.groupby(["activity","subject"]).size().unstack(fill_value=0))


In [ ]:

# Feature extraction
STAT_NAMES = ["mean","std","min","max","rms","q10","q90"]

def _synthetic_raw(activity, subject, segment):
    """
    Deterministic surrogate only for smoke verification.
    Shape and 5x9 sensor organization match the external dataset.
    """
    local = np.random.default_rng(SEED + 10000*activity + 100*subject + segment)
    t = np.linspace(0, 5, 125, endpoint=False)
    arr = np.zeros((125,45), dtype=float)
    # Activity-specific frequency and amplitude; richer peripheral sensors separate classes more.
    freq = 0.35 + 0.11*activity
    base_amp = 0.4 + 0.07*activity
    for unit in range(5):
        for axis in range(9):
            j = 9*unit + axis
            amp = base_amp * (1 + 0.12*unit + 0.02*axis)
            phase = 0.17*axis + 0.11*subject
            arr[:,j] = (
                amp*np.sin(2*np.pi*(freq+0.015*axis)*t + phase)
                + 0.2*np.cos(2*np.pi*(0.15+0.02*unit)*t)
                + local.normal(0, 0.35 + 0.02*unit, size=t.size)
            )
    return arr

def load_raw(row):
    if SMOKE_TEST:
        return _synthetic_raw(int(row.activity), int(row.subject), int(row.segment))
    x = np.loadtxt(row.path, delimiter=",")
    if x.shape != (125,45):
        # Be permissive if whitespace/comma handling differs.
        x = np.genfromtxt(row.path, delimiter=",")
    if x.shape[1] != 45:
        raise ValueError(f"Expected 45 sensor columns, got {x.shape} for {row.path}")
    return x

def summarize_columns(x):
    # x: time x axes
    parts = [
        np.mean(x, axis=0),
        np.std(x, axis=0, ddof=1),
        np.min(x, axis=0),
        np.max(x, axis=0),
        np.sqrt(np.mean(x*x, axis=0)),
        np.quantile(x, 0.10, axis=0),
        np.quantile(x, 0.90, axis=0),
    ]
    return np.concatenate(parts)

def make_level_features(index_df):
    feat_by_level = {l: [] for l in LEVEL_COLS}
    meta = []
    t0 = time.time()
    for i, row in enumerate(index_df.itertuples(index=False), 1):
        raw = load_raw(row)
        for level, cols in LEVEL_COLS.items():
            feat_by_level[level].append(summarize_columns(raw[:, cols]))
        meta.append((int(row.activity), int(row.subject), int(row.segment)))
        if i % 500 == 0:
            print(f"processed {i}/{len(index_df)} segments")
    meta = pd.DataFrame(meta, columns=["activity","subject","segment"])
    feat_by_level = {l: np.asarray(v, dtype=float) for l,v in feat_by_level.items()}
    print(f"Feature extraction finished in {time.time()-t0:.1f}s")
    return meta, feat_by_level

META, XLEVEL = make_level_features(INDEX)
print({l: X.shape for l,X in XLEVEL.items()})


In [ ]:

# Class-conditional Gaussian observation models with train-subject fitting
class GaussianLevelModel:
    def __init__(self, n_components=12, var_floor=1e-3):
        self.n_components = n_components
        self.var_floor = var_floor

    def fit(self, X, y):
        self.classes_ = np.array(sorted(np.unique(y)))
        self.scaler_ = StandardScaler().fit(X)
        Z0 = self.scaler_.transform(X)
        ncomp = min(self.n_components, Z0.shape[1], max(2, Z0.shape[0]-len(self.classes_)-1))
        self.pca_ = PCA(n_components=ncomp, random_state=SEED).fit(Z0)
        Z = self.pca_.transform(Z0)
        self.mu_ = {}
        self.var_ = {}
        for c in self.classes_:
            zc = Z[y == c]
            self.mu_[c] = zc.mean(axis=0)
            self.var_[c] = np.maximum(zc.var(axis=0, ddof=1), self.var_floor)
        return self

    def loglik(self, X):
        X = np.atleast_2d(X)
        Z = self.pca_.transform(self.scaler_.transform(X))
        out = np.empty((len(Z), len(self.classes_)))
        for j,c in enumerate(self.classes_):
            mu, var = self.mu_[c], self.var_[c]
            out[:,j] = -0.5*np.sum(np.log(2*np.pi*var) + (Z-mu)**2/var, axis=1)
        return out

def fit_models(activity_subset):
    mask_train = META.subject.isin(TRAIN_SUBJECTS) & META.activity.isin(activity_subset)
    ytrain = META.loc[mask_train, "activity"].to_numpy()
    models = {}
    for level in LEVEL_COLS:
        models[level] = GaussianLevelModel().fit(XLEVEL[level][mask_train], ytrain)
    return models

BINARY_MODELS = fit_models(BINARY_ACTIVITIES)
MULTI_MODELS = fit_models(MULTI_ACTIVITIES)

print("Binary classes:", BINARY_MODELS[1].classes_)
print("Multi classes:", MULTI_MODELS[1].classes_)


In [ ]:

# Sequential replay engine and uncertainty helpers
def wilson_interval(k, n, confidence=0.95):
    if n == 0:
        return (np.nan, np.nan)
    z = norm.ppf(1-(1-confidence)/2)
    phat = k/n
    den = 1 + z*z/n
    center = (phat + z*z/(2*n))/den
    half = z*np.sqrt(phat*(1-phat)/n + z*z/(4*n*n))/den
    return max(0,center-half), min(1,center+half)

def mean_ci(x, confidence=0.95):
    x = np.asarray(x, dtype=float)
    if len(x) < 2:
        return (float(np.mean(x)), np.nan, np.nan)
    m = x.mean()
    se = x.std(ddof=1)/np.sqrt(len(x))
    z = norm.ppf(1-(1-confidence)/2)
    return m, m-z*se, m+z*se

def softmax_from_log(v):
    return np.exp(v - logsumexp(v))

def sequential_trial(true_class, models, activity_subset, delta, method, rng,
                     fixed_upgrade_every=6, max_steps=MAX_STEPS):
    classes = models[1].classes_
    class_to_idx = {c:i for i,c in enumerate(classes)}
    true_idx = class_to_idx[true_class]

    test_mask = (META.subject.isin(TEST_SUBJECTS)) & (META.activity == true_class)
    candidates = np.flatnonzero(test_mask)
    if len(candidates) == 0:
        raise RuntimeError(f"No test segments for class {true_class}")

    K = len(classes)
    A = math.log((K-1)/delta) if K > 2 else math.log(1/delta)
    gate_thresholds = [rho*A for rho in RHO]

    cum = np.zeros(K)
    level = 1
    max_level = 1
    stage_counts = np.zeros(5, dtype=int)
    total_sensor_cost = 0.0
    unlock_times = [np.nan]*4

    # Shuffle then recycle if the stopping time exceeds available unique segments.
    perm = rng.permutation(candidates)
    ptr = 0

    for t in range(1, max_steps+1):
        if ptr >= len(perm):
            perm = rng.permutation(candidates)
            ptr = 0
        idx = perm[ptr]
        ptr += 1

        if method == "oracle_all":
            level = 5
        elif method == "weak_only":
            level = 1
        elif method == "fixed_schedule":
            level = min(5, 1 + (t-1)//fixed_upgrade_every)
        elif method == "reversible_posterior":
            # Reversible comparator: current accumulated posterior can move sensing level up or down.
            if t > 1:
                conf = np.max(softmax_from_log(cum))
                cuts = [0.65,0.75,0.85,0.93]
                level = 1 + sum(conf >= c for c in cuts)
                level = min(level,5)
        elif method == "proposed_gated":
            level = max_level
        else:
            raise ValueError(method)

        ll = models[level].loglik(XLEVEL[level][idx:idx+1])[0]
        cum += ll
        stage_counts[level-1] += 1
        total_sensor_cost += LEVEL_COST[level]

        order = np.argsort(cum)
        best, second = order[-1], order[-2]
        margin = cum[best] - cum[second]

        if method == "proposed_gated":
            while max_level < 5 and margin >= gate_thresholds[max_level-1]:
                unlock_times[max_level-1] = t
                max_level += 1
            level = max_level

        if margin >= A:
            pred = classes[best]
            return {
                "true": int(true_class), "pred": int(pred), "error": int(pred != true_class),
                "tau": t, "sensor_cost": total_sensor_cost,
                "final_level": int(level), "stage_counts": stage_counts.tolist(),
                "unlock_times": unlock_times, "capped": 0
            }

    pred = classes[int(np.argmax(cum))]
    return {
        "true": int(true_class), "pred": int(pred), "error": int(pred != true_class),
        "tau": max_steps, "sensor_cost": total_sensor_cost,
        "final_level": int(level), "stage_counts": stage_counts.tolist(),
        "unlock_times": unlock_times, "capped": 1
    }

def run_benchmark(models, activity_subset, deltas, n_trials_per_class, tag):
    methods = ["weak_only","oracle_all","fixed_schedule","reversible_posterior","proposed_gated"]
    trial_rows = []
    for delta in deltas:
        for method in methods:
            for c in activity_subset:
                for r in range(n_trials_per_class):
                    rrng = np.random.default_rng(SEED + int(delta*1e6) + 1000*c + 17*r + 31*methods.index(method))
                    out = sequential_trial(c, models, activity_subset, delta, method, rrng)
                    out.update({"delta":delta,"method":method,"rep":r,"benchmark":tag})
                    trial_rows.append(out)
    trials = pd.DataFrame(trial_rows)

    summary = []
    for keys,g in trials.groupby(["benchmark","delta","method"]):
        errors = int(g.error.sum())
        lo,hi = wilson_interval(errors,len(g))
        mt,lmt,hmt = mean_ci(g.tau)
        mc,lmc,hmc = mean_ci(g.sensor_cost)
        summary.append({
            "benchmark":keys[0], "delta":keys[1], "method":keys[2],
            "n_trials":len(g), "error_rate":errors/len(g),
            "error_ci_lo":lo, "error_ci_hi":hi,
            "mean_tau":mt, "tau_ci_lo":lmt, "tau_ci_hi":hmt,
            "median_tau":float(g.tau.median()), "p95_tau":float(g.tau.quantile(.95)),
            "mean_sensor_cost":mc, "sensor_cost_ci_lo":lmc, "sensor_cost_ci_hi":hmc,
            "cap_rate":float(g.capped.mean()),
        })
    return trials, pd.DataFrame(summary)


In [ ]:

# Experiment 1 — Binary real-data benchmark: ascending vs descending stairs
binary_trials, binary_summary = run_benchmark(
    BINARY_MODELS, BINARY_ACTIVITIES, DELTAS, N_BINARY_TRIALS, "binary_stairs"
)
binary_trials.to_json(RESULTS/"external_binary_trials.json", orient="records")
binary_summary.to_csv(RESULTS/"external_binary_summary.csv", index=False)

display(binary_summary.round(4))

# Error-cost plot
fig, ax = plt.subplots(figsize=(7,5))
for method, g in binary_summary.groupby("method"):
    ax.plot(g["mean_sensor_cost"], g["error_rate"], marker="o", label=method)
ax.set_xlabel("Mean cumulative sensor cost")
ax.set_ylabel("Terminal error rate")
ax.set_title("External binary replay: error–sensor-cost trade-off")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES/"external_binary_error_cost.png", dpi=180)
plt.show()


In [ ]:

# Experiment 2 — Six-class external benchmark
multi_trials, multi_summary = run_benchmark(
    MULTI_MODELS, MULTI_ACTIVITIES, DELTAS, N_MULTI_TRIALS, "six_class"
)
multi_trials.to_json(RESULTS/"external_sixclass_trials.json", orient="records")
multi_summary.to_csv(RESULTS/"external_sixclass_summary.csv", index=False)

display(multi_summary.round(4))

# Per-class proposed-gated results at the strictest delta in this run
dstar = min(DELTAS)
pg = multi_trials[(multi_trials.method=="proposed_gated") & (multi_trials.delta==dstar)].copy()
per_class = []
for c,g in pg.groupby("true"):
    k = int(g.error.sum())
    lo,hi = wilson_interval(k,len(g))
    per_class.append({
        "activity":int(c), "activity_name":ACTIVITY_NAMES[int(c)], "n":len(g),
        "error_rate":k/len(g), "error_ci_lo":lo, "error_ci_hi":hi,
        "mean_tau":g.tau.mean(), "mean_sensor_cost":g.sensor_cost.mean(),
        "cap_rate":g.capped.mean()
    })
per_class = pd.DataFrame(per_class)
per_class.to_csv(RESULTS/"external_sixclass_per_class.csv", index=False)
display(per_class.round(4))


In [ ]:

# Experiment 3 — Confusion matrix + stage-usage diagnostics for proposed gated sensing
dstar = min(DELTAS)
pg = multi_trials[(multi_trials.method=="proposed_gated") & (multi_trials.delta==dstar)].copy()
labels = MULTI_ACTIVITIES
cm = confusion_matrix(pg.true, pg.pred, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[ACTIVITY_NAMES[x] for x in labels],
    columns=[ACTIVITY_NAMES[x] for x in labels]
)
cm_df.to_csv(RESULTS/"external_sixclass_confusion.csv")
display(cm_df)

fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(cm)
ax.set_xticks(range(len(labels)), [ACTIVITY_NAMES[x] for x in labels], rotation=45, ha="right")
ax.set_yticks(range(len(labels)), [ACTIVITY_NAMES[x] for x in labels])
ax.set_xlabel("Predicted activity")
ax.set_ylabel("True activity")
ax.set_title(f"Six-class proposed-gated confusion matrix (delta={dstar})")
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j,i,str(cm[i,j]),ha="center",va="center",fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
fig.savefig(FIGURES/"external_sixclass_confusion.png", dpi=180)
plt.show()

# Stage occupancy
stage_mat = np.array(pg.stage_counts.tolist(), dtype=float)
stage_share = stage_mat.sum(axis=0) / max(1,stage_mat.sum())
stage_df = pd.DataFrame({"level":np.arange(1,6),"sample_share":stage_share})
stage_df.to_csv(RESULTS/"external_gated_stage_occupancy.csv", index=False)

fig, ax = plt.subplots(figsize=(6,4))
ax.bar(stage_df.level, stage_df.sample_share)
ax.set_xlabel("Sensing level")
ax.set_ylabel("Fraction of sequential samples")
ax.set_title("Where the gated learner spends its sensing budget")
ax.set_xticks(range(1,6))
fig.tight_layout()
fig.savefig(FIGURES/"external_stage_occupancy.png", dpi=180)
plt.show()


In [ ]:

# Experiment 4 — Delta sensitivity and method ranking
combined = pd.concat([binary_summary, multi_summary], ignore_index=True)
combined.to_csv(RESULTS/"external_all_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(7,5))
g = multi_summary[multi_summary.method.isin(["weak_only","oracle_all","proposed_gated"])]
for method, s in g.groupby("method"):
    s = s.sort_values("delta")
    ax.plot(s.delta, s.mean_tau, marker="o", label=method)
ax.set_xscale("log")
ax.invert_xaxis()
ax.set_xlabel("delta (stricter confidence → right)")
ax.set_ylabel("Mean stopping time")
ax.set_title("Six-class external replay: confidence sensitivity")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES/"external_delta_sensitivity.png", dpi=180)
plt.show()

# Compact reviewer-facing ranking table at strictest delta.
strict = combined[combined.delta == min(DELTAS)].copy()
strict["error_x100"] = 100*strict.error_rate
rank_cols = ["benchmark","method","n_trials","error_x100","mean_tau","mean_sensor_cost","cap_rate"]
ranking = strict[rank_cols].sort_values(["benchmark","error_x100","mean_sensor_cost"])
ranking.to_csv(RESULTS/"external_reviewer_ranking.csv", index=False)
display(ranking.round(3))


In [ ]:

# Experiment 5 — Sanity checks / reviewer closure assertions
checks = []

# 1. Data separation: no test subject in training.
checks.append(("held_out_subject", not set(TRAIN_SUBJECTS).intersection(TEST_SUBJECTS)))

# 2. Richer levels must have strictly more raw sensor axes.
dims = [len(LEVEL_COLS[l]) for l in range(1,6)]
checks.append(("nested_sensor_ladder", all(a < b for a,b in zip(dims[:-1],dims[1:]))))

# 3. Oracle cost/sample should generally exceed weak per sample; cumulative may be lower due to earlier stop.
checks.append(("all_methods_present_binary", set(binary_summary.method) == {
    "weak_only","oracle_all","fixed_schedule","reversible_posterior","proposed_gated"
}))
checks.append(("all_methods_present_multiclass", set(multi_summary.method) == {
    "weak_only","oracle_all","fixed_schedule","reversible_posterior","proposed_gated"
}))

check_df = pd.DataFrame(checks, columns=["check","passed"])
check_df.to_csv(RESULTS/"external_sanity_checks.csv", index=False)
display(check_df)

if not check_df.passed.all():
    raise AssertionError("One or more external benchmark sanity checks failed.")

run_metadata = {
    "dataset": "UCI Daily and Sports Activities",
    "doi": UCI_DOI,
    "real_data": bool(REAL_DATA),
    "smoke_test": bool(SMOKE_TEST),
    "full_run": bool(FULL_RUN),
    "train_subjects": TRAIN_SUBJECTS,
    "calibration_subjects": CAL_SUBJECTS,
    "test_subjects": TEST_SUBJECTS,
    "multi_activities": MULTI_ACTIVITIES,
    "binary_activities": BINARY_ACTIVITIES,
    "n_binary_trials_per_class": N_BINARY_TRIALS,
    "n_multi_trials_per_class": N_MULTI_TRIALS,
    "deltas": DELTAS,
    "max_steps": MAX_STEPS,
    "seed": SEED,
}
(RESULTS/"external_run_metadata.json").write_text(json.dumps(run_metadata, indent=2))
print("All sanity checks passed.")



## How to report these results in the paper

**Main manuscript (recommended):**
- theorem-aligned binary stairs benchmark: error, stopping time, sensor cost
- six-class proposed vs weak-only vs all-sensor oracle
- stage occupancy / sensor-cost trade-off
- held-out-subject design

**Supplement:**
- full per-class confusion matrix
- fixed-schedule and reversible-posterior baselines
- all delta sweeps
- trial-level JSON

Suggested wording:

> “We further instantiated evidence-gated acquisition on the UCI Daily and Sports Activities dataset using a five-level nested sensing ladder induced by the five body-worn sensor units. Models were fit on subjects 1–6 and evaluated on held-out subject 8. The experiment is an offline resource-gated sensing replay rather than a physical safety validation.”

Do **not** claim that this dataset proves the binary nonasymptotic theorem for arbitrary multi-class sensing.


In [ ]:

# FINAL CELL — package and download every Notebook 07 result
bundle = ROOT / "notebook07_external_results.zip"
if bundle.exists():
    bundle.unlink()

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in [RESULTS, FIGURES]:
        for fp in sorted(folder.rglob("*")):
            if fp.is_file():
                zf.write(fp, arcname=str(fp.relative_to(ROOT)))

    # Include a small citation/readme inside the result bundle.
    citation_text = f"""Dataset: UCI Daily and Sports Activities
Creators: Billur Barshan, Kerem Altun
DOI: {UCI_DOI}
Official UCI URL: https://archive.ics.uci.edu/dataset/256/daily
Benchmark type: offline evidence-gated multi-sensor replay
"""
    citation_fp = ROOT / "NOTEBOOK07_DATASET_CITATION.txt"
    citation_fp.write_text(citation_text)
    zf.write(citation_fp, arcname=citation_fp.name)

print(f"Created: {bundle.resolve()}")
print(f"Bundle size: {bundle.stat().st_size/1024:.1f} KB")
print("Files:", len(zipfile.ZipFile(bundle).namelist()))

try:
    from google.colab import files
    files.download(str(bundle))
except Exception:
    print("Not running in Colab. Download the ZIP from the path printed above.")
